# Hello Lucky — YOLOv8n 네잎클로버 감지 학습 (Colab)

이 노트북은 `merge_datasets.py` 가 만든 `merged/` 데이터셋으로
가벼운 YOLOv8-nano 물체감지 모델을 학습합니다.

**시작 전 반드시:** 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택.

**흐름**
1. ultralytics 설치
2. `merged.zip` 업로드 (로컬 `training/merged/` 를 zip 으로 압축한 것)
3. 학습 (YOLOv8n, ~수십 분)
4. 성능 확인 (mAP / precision / recall)
5. 검증 이미지에 실제 예측을 그려 눈으로 확인
6. TFLite / ONNX 로 내보내기 (온디바이스 또는 재업로드용)

**현실적 기대치:** 공개 데이터 900장 남짓, 상당수가 근접샷입니다.
이 모델은 "동작하는 파이프라인 + 데모 수준" 이지 상용 정확도가 아닙니다.
실사용 각도의 필드 데이터를 모아 재학습해야 쓸만해집니다.

## 1. 설치

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()  # GPU 인식 여부 확인

## 2. 데이터셋 업로드

로컬에서 `training/merged/` 폴더를 `merged.zip` 으로 압축한 뒤
아래 셀을 실행하고 그 zip 을 선택해 업로드하세요.

> Windows PowerShell 에서 압축:
> `Compress-Archive -Path training/merged/* -DestinationPath merged.zip`

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # merged.zip 선택
zip_name = next(iter(uploaded))

os.makedirs('/content/merged', exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/merged')

# data.yaml 위치 자동 탐색 (zip 내부 구조가 한 겹 더 있어도 대응)
import glob
yaml_candidates = glob.glob('/content/merged/**/data.yaml', recursive=True)
assert yaml_candidates, 'data.yaml 을 찾지 못했습니다. zip 구조를 확인하세요.'
DATA_YAML = yaml_candidates[0]
print('data.yaml:', DATA_YAML)
print(open(DATA_YAML).read())

## 3. 학습

- `epochs` 는 소규모 데이터라 100 정도로 시작. 과적합되면 줄이세요.
- `imgsz=640` 기본. 작은 클로버가 문제면 `imgsz=1280` 로 올리되 느려집니다.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # nano = 폰에서 돌릴 만큼 가벼움

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,          # 20 epoch 개선 없으면 조기 종료
    project='clover',
    name='yolov8n',
    exist_ok=True,
)

## 4. 성능 확인

`mAP50` 가 실질 지표입니다. 소규모/근접 데이터라 낮게 나와도 정상입니다.
중요한 건 절대 점수보다 **필드 데이터를 추가할 때 오르는 추세**입니다.

In [ ]:
metrics = model.val()
print('mAP50    :', round(float(metrics.box.map50), 4))
print('mAP50-95 :', round(float(metrics.box.map), 4))
print('precision:', round(float(metrics.box.mp), 4))
print('recall   :', round(float(metrics.box.mr), 4))

## 5. 예측 눈으로 확인

숫자만 믿지 말고 실제 박스를 보세요. 세잎을 네잎으로 잡는지가 핵심입니다.

In [ ]:
import glob
from IPython.display import Image, display

val_images = glob.glob('/content/merged/**/valid/images/*.*', recursive=True)[:6]
if not val_images:
    val_images = glob.glob('/content/merged/**/test/images/*.*', recursive=True)[:6]

pred = model.predict(val_images, conf=0.25, save=True, project='clover', name='pred', exist_ok=True)
for p in glob.glob('clover/pred/*.jpg')[:6]:
    display(Image(filename=p, width=420))

## 6. 내보내기 & 다운로드

- **TFLite**: 폰 온디바이스 추론용 (`react-native-fast-tflite`).
- **best.pt**: 나중에 재학습하거나 Roboflow 에 올릴 원본 가중치.
- **ONNX**: 다른 런타임이 필요할 때 대비용.

In [ ]:
from google.colab import files

best = 'clover/yolov8n/weights/best.pt'
trained = YOLO(best)

tflite_path = trained.export(format='tflite')   # int8 양자화가 필요하면 int8=True
onnx_path   = trained.export(format='onnx')

print('TFLite:', tflite_path)
print('ONNX  :', onnx_path)

files.download(best)
files.download(str(tflite_path))

## 다음 단계

**빠른 경로 (앱 코드 수정 없음):** `best.pt` 를 Roboflow 프로젝트에 업로드해
hosted 모델로 배포 → 앱 `.env` 의 `EXPO_PUBLIC_ROBOFLOW_DETECTION_MODEL_ID`
를 채우면 바로 실기기 테스트 가능.

**온디바이스 경로 (비용 0, 오프라인):** `best_float32.tflite` 를 앱에 넣고
`react-native-fast-tflite` 로 VisionCamera 프레임 프로세서에서 직접 추론.

**둘 다 공통:** 실사용 각도로 필드 데이터를 모아 재학습하는 것이
정확도를 올리는 유일한 길입니다.